In [ ]:
# =============================================================================
# CELL 1 - COLAB ENVIRONMENT SETUP
# =============================================================================
# Installs, GPU check, Google Drive mount and Hugging Face login.
# facebook/sam3 is a GATED model: you must accept the license at
# https://huggingface.co/facebook/sam3 with the same account as your token.
# =============================================================================

!pip install -q -U transformers accelerate huggingface_hub supervision

import torch
import torchvision

print("PyTorch version    :", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("CUDA available     :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU                :", torch.cuda.get_device_name(0))
    print("GPU memory (GB)    :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

from google.colab import drive
drive.mount('/content/drive')

from huggingface_hub import login
login()  # paste your HF token (needs access to facebook/sam3)

print("Environment ready.")

In [ ]:
# =============================================================================
# CELL 2 - IMPORTS
# =============================================================================

import os
import gc
import csv
import time
import json
import hashlib

import numpy as np
import pandas as pd
import torch

from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.lines import Line2D

import supervision as sv
from supervision.metrics import MeanAveragePrecision

from transformers import Sam3Model, Sam3Processor

# UAV images are very large; disable PIL's decompression-bomb guard.
Image.MAX_IMAGE_PIXELS = None

print("Imports OK. supervision version:", sv.__version__)

In [ ]:
# =============================================================================
# CELL 3 - CONFIGURATION
# =============================================================================
# Every parameter of the study lives here so that a thesis experiment is fully
# described by this single cell.
#
# PIPELINE OF THIS EXPERIMENT (kept from E04 / the single-image test)
#   the WHOLE UAV image is downscaled so that its longest side = MAX_DIM,
#   ONE SAM3 forward pass per (image, anchor) receives, at the same time,
#     - K_POSITIVES GT Rumex boxes of the SAME image as POSITIVE box prompts
#       (label 1): the anchor + K_POSITIVES-1 seeded random other GT boxes
#     - J_NEGATIVES automatically generated NON-Rumex boxes (label 0),
#       generated anew for every run (seeded)
#   and NO text prompt (visual box prompts only),
#   (at their real location, no exemplar strip, no tiles),
#   the predicted boxes are scaled back to the original resolution.
#
# NEGATIVE EXEMPLARS (CELL 11), generated automatically for every run:
#   GT Rumex boxes
#     -> generate candidate regions (sizes drawn from this image's GT boxes)
#     -> remove candidates overlapping Rumex (0 overlap, gap >= NEG_MIN_GAP_PX)
#     -> prefer candidates close to Rumex (gap <= NEG_MAX_GAP_FACTOR x median
#        plant size), to avoid picking bare soil
#     -> select J_NEGATIVES spatially diverse ones (farthest-point sampling)
#     -> SAM3
# NOTE: the generator uses ALL GT boxes of the image (that is what guarantees
# zero overlap), i.e. label information of the evaluated plants.
# =============================================================================

# ------------------------- experiment identity -------------------------------
EXPERIMENT_NAME = "pos_neg_exemplar_no_tiling"   # K pos + J neg boxes (no text), whole image
# EXPERIMENT_DIR_NAME is the file-system-safe version of the name (a "/" would be
# read as a sub-folder), used ONLY for folder names.
EXPERIMENT_DIR_NAME = EXPERIMENT_NAME.replace("/", "-")
K_POSITIVES     = 2           # positive box prompts per run (anchor + 1 random GT box)
J_NEGATIVES     = 3           # negative box prompts per run (generated, non-Rumex)
N_EXEMPLARS     = K_POSITIVES # positive exemplars = the GT boxes used as prompts
USE_TILING      = False       # whole image, no tiles
PROMPT_TYPE     = f"{K_POSITIVES}pos+{J_NEGATIVES}neg"

# ------------------------------- dataset -------------------------------------
IMAGES_ROOT = "/content/drive/MyDrive/master_thesis/dataset/AGS_Multi_Rumex/images"
LABELS_ROOT = "/content/drive/MyDrive/master_thesis/dataset/AGS_Multi_Rumex/annotations_yolo"
RUMEX_CLASS_ID = 0
VALID_IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png")

# ------------------------------- outputs -------------------------------------
RESULTS_ROOT = f"/content/drive/MyDrive/master_thesis/results_new1/{EXPERIMENT_DIR_NAME}"

# --------------------------- whole-image input -------------------------------
MAX_DIM = 1024                # longest side of the downscaled copy fed to SAM3.
                              # Keeps the mask upsampling inside the SAM3
                              # post-processing cheap (same rule as notebook 1).

# ------------------------ negative exemplar generator -------------------------
NEG_NUM_CANDIDATES = 2000     # random candidate boxes generated over the image
NEG_MIN_GAP_PX     = 10       # a negative must stay at least this far (px, full-res)
                              # from EVERY Rumex GT box -> zero overlap guaranteed
NEG_MAX_GAP_FACTOR = 1.0      # "close to Rumex": gap <= factor x median plant size
                              # (doubled automatically if too few candidates qualify)

# --------------------------- SAM3 inference ----------------------------------
# The confidence threshold is FIXED from the start. It is used both
#   (a) inside SAM3 post-processing (single inference pass per image x anchor), and
#   (b) as the operating point for precision / recall / F1 / IoU1 / IoU2.
# Because (a) == (b), AP and the operating-point metrics use the same predictions.
CONFIDENCE_THRESHOLD = 0.30
MASK_THRESHOLD = 0.40         # SAM3 mask binarisation. No plausibility filter is used
                              # in this experiment, so masks do not influence any
                              # box or metric; they are discarded right after
                              # post-processing.
KEEP_MASKS = False            # masks are NEVER stored (RAM / GPU / disk / CSV / NPZ)

# ------------------------------- NMS -----------------------------------------
NMS_IOU_THRESHOLD = 0.40      # fixed from the start, applied offline (CELL 16)

# ------------------------------ evaluation ------------------------------------
EVAL_IOU_THRESHOLD = 0.50     # IoU needed for a prediction to count as a TP
PROMPT_IGNORE_IOU  = 0.50     # held_out mode: an unmatched prediction whose best IoU
                              # with a PROMPT GT box is >= this value is IGNORED
                              # (neither TP nor FP). Same value as EVAL_IOU_THRESHOLD
                              # so a single IoU threshold governs the whole protocol.

EVALUATION_MODES = ["all_gt", "held_out"]
# all_gt   : every GT box of the image is evaluated (classical evaluation).
# held_out : the K_POSITIVES GT instances used as positive prompts are IGNORED, and
#            so are the predictions that fall on them. Answers "how well does SAM3
#            find the REMAINING Rumex plants after being shown K examples?".
#            (Negatives are not GT boxes, so they do not change the evaluation.)

# --------------------------- qualitative plot ---------------------------------
PLOT_EVALUATION_MODE = "all_gt"   # image-level AP50 of this mode selects the image
PLOT_MIN_GT_BOXES    = 7          # the plotted image must have at least this many GT boxes
PLOT_MAX_DISPLAY_DIM = 2048       # display-only downscale (the full image is 8192 px wide)
PLOT_SHOW_SCORES     = True       # write the confidence next to each predicted box

print(f"Configuration loaded for experiment '{EXPERIMENT_NAME}'")
print(f"  results folder : {RESULTS_ROOT}")
print(f"  prompts        : {K_POSITIVES} positive + {J_NEGATIVES} negative boxes, "
      f"no text ({PROMPT_TYPE})")
print(f"  negatives      : {NEG_NUM_CANDIDATES} candidates, gap >= {NEG_MIN_GAP_PX}px, "
      f"close = gap <= {NEG_MAX_GAP_FACTOR} x median plant size, spatially diverse, new per run")
print(f"  tiling         : {USE_TILING} (whole image, longest side -> {MAX_DIM} px)")
print(f"  confidence     : {CONFIDENCE_THRESHOLD} (fixed, single inference pass)")
print(f"  mask threshold : {MASK_THRESHOLD}")
print(f"  NMS IoU        : {NMS_IOU_THRESHOLD} (fixed)")
print(f"  masks stored   : {KEEP_MASKS}")

In [ ]:
# =============================================================================
# CELL 4 - OUTPUT FOLDERS
# =============================================================================
# results/<EXPERIMENT_DIR_NAME>/
#   raw_detections/      pre-NMS detections (NPZ, one file per image x anchor)
#   metrics/             run / image / experiment / dataset level CSVs
#   confusion_matrices/  CSV + PNG for all_gt and held_out
#   plots/               qualitative GT-vs-prediction figure (CELL 26)
# =============================================================================

RAW_DETECTIONS_DIR    = os.path.join(RESULTS_ROOT, "raw_detections")
METRICS_DIR           = os.path.join(RESULTS_ROOT, "metrics")
CONFUSION_MATRIX_DIR  = os.path.join(RESULTS_ROOT, "confusion_matrices")
PLOTS_DIR             = os.path.join(RESULTS_ROOT, "plots")

for d in [RESULTS_ROOT, RAW_DETECTIONS_DIR, METRICS_DIR,
          CONFUSION_MATRIX_DIR, PLOTS_DIR]:
    os.makedirs(d, exist_ok=True)

# Manifest of finished inference runs -> used for crash-safe resuming.
RUN_MANIFEST_CSV = os.path.join(RAW_DETECTIONS_DIR, "runs_manifest.csv")
MANIFEST_COLUMNS = [
    "experiment_name", "image_ID", "anchor_idx", "Prompt_ID", "Prompt_Type",
    "n_gt", "n_prompt_gt", "n_negatives", "neg_max_gap_px", "n_detections_pre_nms",
    "image_width", "image_height", "npz_file", "inference_seconds",
]

print("Output folders ready under:", RESULTS_ROOT)

In [ ]:
# =============================================================================
# CELL 5 - SAM3 MODEL + PROCESSOR
# =============================================================================

device = "cuda" if torch.cuda.is_available() else "cpu"

sam3_model = Sam3Model.from_pretrained("facebook/sam3", device_map="auto")
sam3_model.eval()

sam3_processor = Sam3Processor.from_pretrained("facebook/sam3")

print("SAM3 loaded.")
print("  model device:", next(sam3_model.parameters()).device)
print("  model dtype :", next(sam3_model.parameters()).dtype)

In [ ]:
# =============================================================================
# CELL 6 - STABLE REPRODUCIBILITY HELPERS
# =============================================================================
# The positive-exemplar selection AND the negative generator are seeded with a
# SHA-256 digest of a plain text key (not Python's hash(), which is randomised per
# process), so every run gets the same prompts in every session / machine.
# With N_EXEMPLARS = K_POSITIVES = 2 every run uses the anchor + 1 random GT box.
# =============================================================================

def stable_seed(*parts) -> int:
    """
    Deterministic 32-bit seed from any set of values.

    Input : any number of values (strings / ints) that identify the run,
            e.g. stable_seed(EXPERIMENT_NAME, image_id, anchor_idx)
    Output: int in [0, 2**32) - identical in every Python process, forever.
    """
    key = "|".join(str(p) for p in parts)
    digest = hashlib.sha256(key.encode("utf-8")).digest()
    return int.from_bytes(digest[:8], "big") % (2 ** 32)


def select_exemplar_indices(n_gt: int,
                            anchor_idx: int,
                            n_exemplars: int,
                            image_id: str,
                            experiment_name: str = EXPERIMENT_NAME) -> list:
    """
    Choose which GT instances of ONE image are used as visual prompts.

    Input : n_gt        - number of GT boxes in the image
            anchor_idx  - index of the GT box this run is "about" (always a prompt)
            n_exemplars - how many positive prompts in total (K_POSITIVES here)
            image_id    - "<folder>/<image name>"
    Output: list of GT indices, ANCHOR FIRST, then the randomly sampled others.
            With n_exemplars = 1 this is simply [anchor_idx].
    """
    seed = stable_seed(experiment_name, image_id, anchor_idx)
    rng = np.random.default_rng(seed)

    others = [i for i in range(n_gt) if i != anchor_idx]
    n_needed = min(n_exemplars - 1, len(others))
    if n_needed > 0:
        chosen = [int(i) for i in rng.choice(others, size=n_needed, replace=False)]
    else:
        chosen = []
    return [int(anchor_idx)] + chosen


def format_prompt_id(exemplar_indices: list) -> str:
    """
    Human-readable id of a prompt set.
      single   -> "5"
      multiple -> "5+12+3"  (the ANCHOR is always the first number)
    """
    return "+".join(str(int(i)) for i in exemplar_indices)


# --- quick self-test: the same key must always give the same seed -------------
_demo = stable_seed(EXPERIMENT_NAME, "folderA/img_001", 3)
print("stable_seed demo :", _demo, "(identical in every session / machine)")
print("exemplar demo    :", select_exemplar_indices(10, 3, N_EXEMPLARS, "folderA/img_001"))

In [ ]:
# =============================================================================
# CELL 7 - DATASET AND YOLO ANNOTATION HELPERS
# =============================================================================

def load_yolo_boxes(label_path, img_width, img_height, class_id=RUMEX_CLASS_ID):
    """
    Read a YOLO .txt annotation file and convert it to pixel corner boxes.

    Input : label_path            - YOLO txt file
            img_width, img_height - size of the ORIGINAL image in pixels
            class_id              - keep only this class (0 = Rumex)
    Output: np.ndarray (N, 4) float32, boxes as [x1, y1, x2, y2] in pixels.

    YOLO stores normalised (class, x_center, y_center, width, height).
    """
    boxes = []
    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue                      # skip empty lines
            if int(parts[0]) != class_id:
                continue                      # keep only the requested class
            xc, yc, bw, bh = map(float, parts[1:5])
            xc, yc = xc * img_width, yc * img_height
            bw, bh = bw * img_width, bh * img_height
            boxes.append([xc - bw / 2, yc - bh / 2, xc + bw / 2, yc + bh / 2])
    return np.array(boxes, dtype=np.float32).reshape(-1, 4)


def find_label_path(image_filename_no_ext, folder_name):
    """
    Locate the YOLO txt belonging to an image, supporting both a mirrored folder
    structure (labels/<folder>/<n>.txt) and a flat one (labels/<n>.txt).
    Returns the path or None.
    """
    mirrored = os.path.join(LABELS_ROOT, folder_name, image_filename_no_ext + ".txt")
    flat = os.path.join(LABELS_ROOT, image_filename_no_ext + ".txt")
    if os.path.exists(mirrored):
        return mirrored
    if os.path.exists(flat):
        return flat
    return None


def discover_images():
    """
    Walk IMAGES_ROOT and collect every image together with its label file.
    Output: list of (folder, image_path, label_path, image_id) tuples,
            where image_id = "<folder>/<image name without extension>".
    """
    records = []
    for folder in sorted(os.listdir(IMAGES_ROOT)):
        folder_path = os.path.join(IMAGES_ROOT, folder)
        if not os.path.isdir(folder_path):
            continue
        for fname in sorted(os.listdir(folder_path)):
            if not fname.lower().endswith(VALID_IMAGE_EXTENSIONS):
                continue
            name_no_ext = os.path.splitext(fname)[0]
            label_path = find_label_path(name_no_ext, folder)
            image_id = f"{folder}/{name_no_ext}"
            records.append((folder, os.path.join(folder_path, fname), label_path, image_id))
    return records


def safe_filename(image_id: str) -> str:
    """'folder/name' -> 'folder__name' so it can be used inside a file name."""
    return image_id.replace("/", "__").replace(os.sep, "__")


image_records = discover_images()
missing_labels = [r for r in image_records if r[2] is None]
valid_images = [r for r in image_records if r[2] is not None]

print(f"Discovered {len(image_records)} images in "
      f"{len(set(r[0] for r in image_records))} folders.")
print(f"With labels: {len(valid_images)} | without labels (skipped): {len(missing_labels)}")
if missing_labels:
    print("  first missing:", [r[3] for r in missing_labels[:5]])

In [ ]:
# =============================================================================
# CELL 8 - IoU
# =============================================================================

def compute_iou_matrix(boxes1, boxes2):
    """
    Pairwise IoU between two sets of [x1, y1, x2, y2] boxes.

    Input : boxes1 (N, 4), boxes2 (M, 4)
    Output: (N, M) matrix, entry [i, j] = IoU(boxes1[i], boxes2[j])
    """
    boxes1 = np.asarray(boxes1, dtype=np.float32).reshape(-1, 4)
    boxes2 = np.asarray(boxes2, dtype=np.float32).reshape(-1, 4)
    if len(boxes1) == 0 or len(boxes2) == 0:
        return np.zeros((len(boxes1), len(boxes2)), dtype=np.float32)

    x1 = np.maximum(boxes1[:, None, 0], boxes2[None, :, 0])   # left
    y1 = np.maximum(boxes1[:, None, 1], boxes2[None, :, 1])   # top
    x2 = np.minimum(boxes1[:, None, 2], boxes2[None, :, 2])   # right
    y2 = np.minimum(boxes1[:, None, 3], boxes2[None, :, 3])   # bottom

    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area1 = (boxes1[:, 2] - boxes1[:, 0]) * (boxes1[:, 3] - boxes1[:, 1])
    area2 = (boxes2[:, 2] - boxes2[:, 0]) * (boxes2[:, 3] - boxes2[:, 1])
    union = area1[:, None] + area2[None, :] - inter
    return np.where(union > 0, inter / union, 0.0).astype(np.float32)


print("IoU helper ready.")

In [ ]:
# =============================================================================
# CELL 9 - CORRECT ONE-TO-ONE MATCHING
# =============================================================================
# THE RULE
#   sort predictions by confidence, high -> low
#   for each prediction:
#       look ONLY at GT boxes that are still unmatched
#       take the unmatched GT with the highest IoU
#       if that IoU >= evaluation IoU threshold -> match, else leave unmatched
#   one GT can be matched by at most one prediction, and vice versa.
#
# (The original single-image notebook took the GLOBAL best GT and discarded the
# prediction if that GT was already taken, even when another free GT had
# IoU >= threshold. That under-counted TPs on images with clustered plants.)
#
# This single function is used for TP / FP / FN / precision / recall / F1 /
# IoU1 / IoU2 / confusion matrices, so the whole thesis uses one definition.
# =============================================================================

def match_one_to_one(pred_boxes, pred_scores, gt_boxes, iou_threshold=EVAL_IOU_THRESHOLD):
    """
    Input : pred_boxes  (P, 4), pred_scores (P,), gt_boxes (G, 4), iou_threshold
    Output: dict with
        'pred_match_gt' (P,) int   - matched GT index per prediction, -1 if unmatched
        'gt_match_pred' (G,) int   - matched prediction index per GT,  -1 if unmatched
        'pred_iou'      (P,) float - IoU of the accepted match, 0.0 if unmatched
        'matched_ious'  list       - IoU values of all accepted matches
    """
    pred_boxes = np.asarray(pred_boxes, dtype=np.float32).reshape(-1, 4)
    gt_boxes = np.asarray(gt_boxes, dtype=np.float32).reshape(-1, 4)
    n_pred, n_gt = len(pred_boxes), len(gt_boxes)

    pred_match_gt = np.full(n_pred, -1, dtype=np.int64)
    gt_match_pred = np.full(n_gt, -1, dtype=np.int64)
    pred_iou = np.zeros(n_pred, dtype=np.float32)   # IoU of the accepted match per prediction

    if n_pred == 0 or n_gt == 0:
        return {"pred_match_gt": pred_match_gt, "gt_match_pred": gt_match_pred,
                "pred_iou": pred_iou, "matched_ious": []}

    iou = compute_iou_matrix(pred_boxes, gt_boxes)
    gt_free = np.ones(n_gt, dtype=bool)             # which GTs are still available

    # 'stable' keeps the original order for equal scores -> fully deterministic.
    order = np.argsort(-np.asarray(pred_scores, dtype=np.float32), kind="stable")

    matched_ious = []
    for p in order:
        if not gt_free.any():
            break                                   # every GT already has a prediction
        # Consider ONLY currently unmatched GT boxes.
        candidate_ious = np.where(gt_free, iou[p], -1.0)
        g = int(np.argmax(candidate_ious))          # best FREE GT
        if candidate_ious[g] >= iou_threshold:
            gt_free[g] = False
            pred_match_gt[p] = g
            gt_match_pred[g] = p
            pred_iou[p] = candidate_ious[g]
            matched_ious.append(float(candidate_ious[g]))

    return {"pred_match_gt": pred_match_gt, "gt_match_pred": gt_match_pred,
            "pred_iou": pred_iou, "matched_ious": matched_ious}


def safe_f1(precision, recall):
    """F1 = 2PR/(P+R) with a safe zero denominator (returns 0.0)."""
    denom = precision + recall
    return float(2.0 * precision * recall / denom) if denom > 0 else 0.0


print("One-to-one matcher ready (unmatched-GT-aware).")

In [ ]:
# =============================================================================
# CELL 10 - RESIZE HELPER (whole-image, no tiling)
# =============================================================================
# The WHOLE image is sent to SAM3 in one pass (no tiles). The large UAV image is
# downscaled so that its longest side is MAX_DIM, purely for SAM3 input size /
# speed (it keeps the mask upsampling inside the post-processing cheap).
# Predictions are rescaled back to full resolution right after inference
# (see CELL 12).
# =============================================================================

def resize_for_sam3(img, max_dim=MAX_DIM):
    w, h = img.size
    scale = max_dim / max(w, h)
    if scale >= 1:
        return img, 1.0
    new_w, new_h = int(w * scale), int(h * scale)
    return img.resize((new_w, new_h), Image.BILINEAR), scale


print(f"Resize helper ready (MAX_DIM={MAX_DIM}).")

In [ ]:
# =============================================================================
# CELL 11 - NEGATIVE EXEMPLAR GENERATOR
# =============================================================================
#   GT Rumex boxes
#         |
#   1. generate NEG_NUM_CANDIDATES candidate regions: random positions inside the
#      image, sizes drawn from this image's own GT boxes (plant-sized regions)
#         |
#   2. remove every candidate that overlaps Rumex: zero intersection with ALL GT
#      boxes AND a gap of at least NEG_MIN_GAP_PX (10 px) to the nearest one
#         |
#   3. prefer candidates close to Rumex (vegetated surroundings, not bare soil):
#      keep gap <= NEG_MAX_GAP_FACTOR x median plant size; if fewer than
#      J_NEGATIVES qualify, the limit is doubled (up to 3 times)
#         |
#   4. select J_NEGATIVES spatially diverse ones by farthest-point sampling on
#      the box centres, starting from the candidate closest to Rumex; selected
#      negatives never overlap each other
#         |
#   SAM3 (label 0 box prompts)
# =============================================================================

def box_gap_matrix(boxes_a, boxes_b):
    """
    Euclidean gap between axis-aligned boxes.
    Input : boxes_a (N,4), boxes_b (M,4) as [x1,y1,x2,y2]
    Output: (N,M) distance in px between the box borders; 0 when they touch/overlap.
    """
    a = np.asarray(boxes_a, dtype=np.float32).reshape(-1, 4)[:, None, :]
    b = np.asarray(boxes_b, dtype=np.float32).reshape(-1, 4)[None, :, :]
    dx = np.maximum(0.0, np.maximum(b[..., 0] - a[..., 2], a[..., 0] - b[..., 2]))
    dy = np.maximum(0.0, np.maximum(b[..., 1] - a[..., 3], a[..., 1] - b[..., 3]))
    return np.sqrt(dx ** 2 + dy ** 2)


def intersection_area_matrix(boxes_a, boxes_b):
    """(N,M) intersection area in px^2 between two sets of [x1,y1,x2,y2] boxes."""
    a = np.asarray(boxes_a, dtype=np.float32).reshape(-1, 4)[:, None, :]
    b = np.asarray(boxes_b, dtype=np.float32).reshape(-1, 4)[None, :, :]
    iw = np.clip(np.minimum(a[..., 2], b[..., 2]) - np.maximum(a[..., 0], b[..., 0]), 0, None)
    ih = np.clip(np.minimum(a[..., 3], b[..., 3]) - np.maximum(a[..., 1], b[..., 1]), 0, None)
    return iw * ih


def generate_negative_exemplars(gt_boxes, img_w, img_h, n_negatives=J_NEGATIVES,
                                seed=0, n_candidates=NEG_NUM_CANDIDATES,
                                min_gap=NEG_MIN_GAP_PX, max_gap_factor=NEG_MAX_GAP_FACTOR):
    """
    Input : gt_boxes (G,4) full-res Rumex boxes, image size, number of negatives,
            deterministic seed and generator parameters
    Output: dict with
              'boxes'      (n,4) selected negative boxes, full-res [x1,y1,x2,y2]
              'gaps'       (n,)  gap of each negative to its nearest Rumex box (px)
              'pool_boxes' (P,4) the close, non-overlapping candidate pool
              'stats'      counters for the log
    """
    gt = np.asarray(gt_boxes, dtype=np.float32).reshape(-1, 4)
    if len(gt) == 0:
        raise ValueError("The image has no Rumex GT boxes -> no negatives can be placed near Rumex.")
    rng = np.random.default_rng(seed)

    # ---- 1. candidate regions (plant-sized) -----------------------------------
    gt_w = gt[:, 2] - gt[:, 0]
    gt_h = gt[:, 3] - gt[:, 1]
    size_idx = rng.integers(0, len(gt), n_candidates)
    cw = np.minimum(gt_w[size_idx], img_w - 1)
    ch = np.minimum(gt_h[size_idx], img_h - 1)
    cx1 = rng.uniform(0, img_w - cw)
    cy1 = rng.uniform(0, img_h - ch)
    cand = np.stack([cx1, cy1, cx1 + cw, cy1 + ch], axis=1).astype(np.float32)

    # ---- 2. no overlap with Rumex (+ minimum gap) -----------------------------
    gap_to_rumex = box_gap_matrix(cand, gt).min(axis=1)
    overlap_to_rumex = intersection_area_matrix(cand, gt).max(axis=1)
    valid = (overlap_to_rumex == 0) & (gap_to_rumex >= min_gap)

    # ---- 3. close to Rumex -----------------------------------------------------
    median_plant = float(np.median(np.maximum(gt_w, gt_h)))
    factor = max_gap_factor
    for _ in range(4):
        max_gap = factor * median_plant
        close = valid & (gap_to_rumex <= max_gap)
        if close.sum() >= n_negatives:
            break
        factor *= 2.0
    pool = np.where(close)[0]
    if len(pool) < n_negatives:                    # last resort: nearest valid ones
        pool = np.where(valid)[0]
    pool = pool[np.argsort(gap_to_rumex[pool], kind="stable")]   # nearest first

    # ---- 4. spatially diverse selection (farthest-point sampling) --------------
    centres = np.stack([(cand[:, 0] + cand[:, 2]) / 2, (cand[:, 1] + cand[:, 3]) / 2], axis=1)
    selected = []
    if len(pool):
        selected.append(int(pool[0]))              # start with the one closest to Rumex
    while len(selected) < n_negatives and len(selected) < len(pool):
        sel = np.array(selected)
        d = np.linalg.norm(centres[pool][:, None, :] - centres[sel][None, :, :], axis=2).min(axis=1)
        clash = intersection_area_matrix(cand[pool], cand[sel]).max(axis=1) > 0
        d[np.isin(pool, sel) | clash] = -1.0      # never re-pick or overlap a negative
        best = int(np.argmax(d))
        if d[best] < 0:
            break
        selected.append(int(pool[best]))

    boxes = cand[selected].reshape(-1, 4)
    # hard guarantee: zero overlap with every Rumex GT box
    assert (intersection_area_matrix(boxes, gt) == 0).all()

    return {
        "boxes": boxes,
        "gaps": gap_to_rumex[selected],
        "pool_boxes": cand[pool].reshape(-1, 4),
        "stats": {
            "n_candidates": int(n_candidates),
            "n_no_overlap": int(valid.sum()),
            "n_close": int(len(pool)),
            "median_plant_px": median_plant,
            "max_gap_px": float(max_gap),
            "n_selected": int(len(selected)),
        },
    }


print(f"Negative exemplar generator ready ({J_NEGATIVES} negatives, gap >= {NEG_MIN_GAP_PX}px).")

In [ ]:
# =============================================================================
# CELL 12 - SAM3 WHOLE-IMAGE INFERENCE (no tiling, positive + negative boxes, no text)
# =============================================================================
# Pipeline of this experiment:
#   - resize the whole image to MAX_DIM
#   - scale the positive (GT) and negative (generated) boxes into that space
#   - single SAM3 forward pass, no tiling, no exemplar-strip composition:
#     positive boxes (label 1) and negative boxes (label 0) are passed in the
#     SAME call -> SAM3 conditions on both at once. No text prompt is given.
#   - scale the predicted boxes back up to full resolution
#
# SAM3 is called ONCE per (image_ID, anchor_idx) at CONFIDENCE_THRESHOLD;
# NMS is applied offline afterwards.
# =============================================================================

def _to_numpy(x):
    """Handles tensors (incl. bf16/fp16), lists of tensors, or plain arrays."""
    if torch.is_tensor(x):
        if x.dtype in (torch.bfloat16, torch.float16):
            x = x.float()
        return x.detach().cpu().numpy()
    if isinstance(x, (list, tuple)):
        if len(x) > 0 and torch.is_tensor(x[0]):
            x = [t.float() if t.dtype in (torch.bfloat16, torch.float16) else t for t in x]
            return torch.stack([t.detach().cpu() for t in x]).numpy()
        return np.array(x)
    return np.array(x)


def run_sam3_whole_image(image: Image.Image, pos_boxes_fullres: list,
                         neg_boxes_fullres: list,
                         threshold: float = CONFIDENCE_THRESHOLD,
                         mask_threshold: float = MASK_THRESHOLD):
    """
    pos_boxes_fullres: list of POSITIVE [x1,y1,x2,y2] boxes (ORIGINAL image pixel coords)
    neg_boxes_fullres: list of NEGATIVE [x1,y1,x2,y2] boxes (ORIGINAL image pixel coords)

    Returns: pred_boxes_fullres (np.ndarray Nx4), pred_scores (np.ndarray N),
             all boxes/scores with score > threshold. Masks are requested from
             SAM3 internally (needed by post-processing) but never stored.
    """
    img_w, img_h = image.size
    image_sam, sam_scale = resize_for_sam3(image, MAX_DIM)

    all_boxes_fullres = list(pos_boxes_fullres) + list(neg_boxes_fullres)
    input_boxes_xyxy = [[float(c) * sam_scale for c in box] for box in all_boxes_fullres]
    input_boxes = [input_boxes_xyxy]                           # [batch=1, num_boxes, 4]
    input_boxes_labels = [[1] * len(pos_boxes_fullres)         # 1 = positive prompt
                          + [0] * len(neg_boxes_fullres)]      # 0 = negative prompt

    inputs = sam3_processor(
        images=image_sam,
        input_boxes=input_boxes,                             # positive/negative boxes only
        input_boxes_labels=input_boxes_labels,
        return_tensors="pt",
    ).to(sam3_model.device)

    with torch.no_grad():
        outputs = sam3_model(**inputs)

    results = sam3_processor.post_process_instance_segmentation(
        outputs, threshold=threshold, mask_threshold=mask_threshold,
        target_sizes=inputs.get("original_sizes").tolist(),
    )[0]

    pred_boxes_np = _to_numpy(results["boxes"]).reshape(-1, 4)
    pred_scores_np = _to_numpy(results["scores"]).reshape(-1)

    # Boxes came back in image_sam's (resized) coordinate space -> rescale to full-res
    if len(pred_boxes_np) > 0:
        pred_boxes_np = pred_boxes_np / sam_scale

    # masks are never kept (KEEP_MASKS = False) -> drop the reference immediately
    del results, outputs, inputs

    return pred_boxes_np, pred_scores_np


print("SAM3 whole-image inference helper ready (no tiling, pos/neg boxes, no text).")

In [ ]:
# =============================================================================
# CELL 13 - PRE-NMS DETECTION STORAGE
# =============================================================================
# For every run (= one image x one anchor) we store the detections AFTER
#   SAM3 inference at CONFIDENCE_THRESHOLD -> conversion to original-image
#   coordinates
# but BEFORE NMS.
# The offline evaluation (CELL 15 onwards) therefore never needs SAM3 again.
# Masks are never stored. NPZ is compact and fast; the metric tables are CSV.
# =============================================================================

def run_npz_path(image_id, anchor_idx):
    """Path of the NPZ holding the pre-NMS detections of one run."""
    return os.path.join(RAW_DETECTIONS_DIR,
                        f"{safe_filename(image_id)}__anchor{int(anchor_idx):03d}.npz")


def save_run_detections(image_id, anchor_idx, boxes, scores, gt_boxes,
                        prompt_indices, neg_boxes, image_size):
    """
    Write one run's pre-NMS detections to NPZ. The file is self-contained: it also
    stores the GT boxes, the positive prompt indices and the negative prompt boxes,
    so the whole offline evaluation (and the final plot) can run without
    re-opening label files or re-generating the negatives.
    """
    path = run_npz_path(image_id, anchor_idx)
    np.savez_compressed(
        path,
        experiment_name=np.array(EXPERIMENT_NAME),
        image_ID=np.array(image_id),
        anchor_idx=np.array(int(anchor_idx)),
        prompt_indices=np.array(prompt_indices, dtype=np.int32),
        neg_boxes=np.asarray(neg_boxes, dtype=np.float32).reshape(-1, 4),
        image_width=np.array(int(image_size[0])),
        image_height=np.array(int(image_size[1])),
        gt_boxes=gt_boxes.astype(np.float32),
        boxes=np.asarray(boxes, dtype=np.float32).reshape(-1, 4),   # x1,y1,x2,y2 (original img)
        scores=np.asarray(scores, dtype=np.float32).reshape(-1),    # confidence > 0.30
    )
    return path


def load_run_detections(path):
    """Read one run NPZ back into a plain python dict."""
    with np.load(path, allow_pickle=False) as z:
        return {
            "image_ID": str(z["image_ID"]),
            "anchor_idx": int(z["anchor_idx"]),
            "prompt_indices": z["prompt_indices"].astype(int),
            "neg_boxes": z["neg_boxes"].reshape(-1, 4),
            "image_width": int(z["image_width"]),
            "image_height": int(z["image_height"]),
            "gt_boxes": z["gt_boxes"].reshape(-1, 4),
            "boxes": z["boxes"].reshape(-1, 4),
            "scores": z["scores"].reshape(-1),
        }


print("Pre-NMS detection storage ready ->", RAW_DETECTIONS_DIR)

In [ ]:
# =============================================================================
# CELL 14 - MAIN GPU INFERENCE LOOP  (PHASE 1)
# =============================================================================
# FOR EACH IMAGE:
#     open the original image ONCE
#     read its GT boxes ONCE
#     FOR EACH anchor (= every GT box, once):
#         select the K_POSITIVES positives deterministically (stable_seed, CELL 6)
#         generate J_NEGATIVES negatives deterministically (CELL 11)
#         run SAM3 ONCE over the whole (resized) image with
#             positive boxes + negative boxes (no text)
#         save the PRE-NMS detections + the prompts (NPZ)
#     release the image
#
# NO NMS and NO metric computation happens here - that is all done offline in
# the following cells. The loop is resumable: finished runs are listed in
# runs_manifest.csv.
# =============================================================================

# ---- resume support ---------------------------------------------------------
done_runs = set()
manifest_exists = os.path.exists(RUN_MANIFEST_CSV)
if manifest_exists:
    _m = pd.read_csv(RUN_MANIFEST_CSV)
    _m = _m[_m["experiment_name"] == EXPERIMENT_NAME]
    other_types = sorted(set(_m["Prompt_Type"].astype(str)) - {PROMPT_TYPE})
    if other_types:
        raise RuntimeError(
            f"{RUN_MANIFEST_CSV} already holds results for another prompt setting "
            f"{other_types}. Use a new EXPERIMENT_NAME for {PROMPT_TYPE} "
            "(or delete the old results folder) so the runs are not mixed.")
    done_runs = set(zip(_m["image_ID"].astype(str), _m["anchor_idx"].astype(int)))
    print(f"Resuming: {len(done_runs)} runs already finished for {EXPERIMENT_NAME}.")

manifest_file = open(RUN_MANIFEST_CSV, "a", newline="")
manifest_writer = csv.DictWriter(manifest_file, fieldnames=MANIFEST_COLUMNS)
if not manifest_exists:
    manifest_writer.writeheader()

start_time = time.time()
n_new_runs = 0
image_times = []
n_total_images = len(valid_images)

try:
    for img_idx, (folder, image_path, label_path, image_id) in enumerate(valid_images, start=1):
        image_t0 = time.time()

        # ---------------- open the original image exactly once ----------------
        image = Image.open(image_path).convert("RGB")
        img_w, img_h = image.size
        gt_boxes = load_yolo_boxes(label_path, img_w, img_h, RUMEX_CLASS_ID)
        n_gt = len(gt_boxes)

        if n_gt == 0:
            print(f"[{EXPERIMENT_NAME}] ({img_idx}/{n_total_images}) {image_id}: 0 GT boxes, skipped.")
            image.close(); del image; gc.collect()
            continue

        # skip the whole image if every anchor is already done
        if all((image_id, a) in done_runs for a in range(n_gt)):
            print(f"[{EXPERIMENT_NAME}] ({img_idx}/{n_total_images}) {image_id}: all "
                  f"{n_gt} anchors already done, skipped.")
            image.close(); del image; gc.collect()
            continue

        for anchor_idx in range(n_gt):                 # every GT box is an anchor once
            if (image_id, anchor_idx) in done_runs:
                continue
            run_t0 = time.time()

            # deterministic positive selection (SHA-256 based, see CELL 6)
            exemplar_indices = select_exemplar_indices(n_gt, anchor_idx, N_EXEMPLARS, image_id)
            prompt_id = format_prompt_id(exemplar_indices)
            pos_boxes_fullres = [gt_boxes[i].tolist() for i in exemplar_indices]

            # deterministic negative generation, new for every run (CELL 11)
            neg = generate_negative_exemplars(
                gt_boxes, img_w, img_h, J_NEGATIVES,
                seed=stable_seed(EXPERIMENT_NAME, image_id, anchor_idx, "negatives"))
            neg_boxes_fullres = neg["boxes"].tolist()

            pred_boxes_np, pred_scores_np = run_sam3_whole_image(
                image, pos_boxes_fullres, neg_boxes_fullres,
                threshold=CONFIDENCE_THRESHOLD, mask_threshold=MASK_THRESHOLD,
            )

            npz_path = save_run_detections(image_id, anchor_idx, pred_boxes_np, pred_scores_np,
                                           gt_boxes, exemplar_indices, neg["boxes"],
                                           (img_w, img_h))

            run_seconds = time.time() - run_t0
            manifest_writer.writerow({
                "experiment_name": EXPERIMENT_NAME,
                "image_ID": image_id,
                "anchor_idx": anchor_idx,
                "Prompt_ID": prompt_id,
                "Prompt_Type": PROMPT_TYPE,
                "n_gt": n_gt,
                "n_prompt_gt": len(exemplar_indices),
                "n_negatives": int(len(neg["boxes"])),
                "neg_max_gap_px": round(neg["stats"]["max_gap_px"], 1),
                "n_detections_pre_nms": int(len(pred_scores_np)),
                "image_width": img_w,
                "image_height": img_h,
                "npz_file": os.path.basename(npz_path),
                "inference_seconds": round(run_seconds, 2),
            })
            manifest_file.flush()                      # this run is on disk -> resumable
            n_new_runs += 1

            print(f"  [{EXPERIMENT_NAME}] run #{n_new_runs} | {image_id} | "
                  f"anchor={anchor_idx} ({anchor_idx + 1}/{n_gt}) | "
                  f"prompt=POS {prompt_id} + {len(neg['boxes'])} NEG | "
                  f"pre-NMS detections={len(pred_scores_np)} | {run_seconds:.1f}s")

            if len(neg["boxes"]) < J_NEGATIVES:
                print(f"    WARNING: only {len(neg['boxes'])} of {J_NEGATIVES} negatives could be placed.")

            del pred_boxes_np, pred_scores_np, neg
            gc.collect()
            if device == "cuda":
                torch.cuda.empty_cache()

        # ---------------- release the image ------------------------------------
        image.close()
        del image
        gc.collect()
        if device == "cuda":
            torch.cuda.empty_cache()

        image_elapsed = time.time() - image_t0
        image_times.append(image_elapsed)
        avg_per_image = float(np.mean(image_times))
        eta = (n_total_images - img_idx) * avg_per_image
        print(f"[{EXPERIMENT_NAME}] ({img_idx}/{n_total_images}) {image_id} done | "
              f"{n_gt} GT box(es) | {image_elapsed:.1f}s | avg/image={avg_per_image:.1f}s | "
              f"ETA={eta / 60:.1f} min ({eta / 3600:.2f} h)")
finally:
    manifest_file.close()                    # also closed if the loop crashes

total_elapsed = time.time() - start_time
print(f"\nInference finished for {EXPERIMENT_NAME}: {n_new_runs} new runs.")
print(f"Total time: {total_elapsed / 60:.1f} min ({total_elapsed / 3600:.2f} h)")
print(f"Pre-NMS detections in: {RAW_DETECTIONS_DIR}")

In [ ]:
# =============================================================================
# CELL 15 - LOAD CACHED PRE-NMS DETECTIONS  (PHASE 2 STARTS HERE)
# =============================================================================
# From here on SAM3 is never touched again. Everything below works on the NPZ
# files written in CELL 14, so you can restart the runtime, free the GPU and
# still redo the complete evaluation in a few minutes.
# To run PHASE 2 only: run CELLS 1-4 and 6-13 (skip CELL 5 and CELL 14), then
# continue from here.
# =============================================================================

manifest = pd.read_csv(RUN_MANIFEST_CSV)
manifest = manifest[(manifest["experiment_name"] == EXPERIMENT_NAME) &
                    (manifest["Prompt_Type"].astype(str) == PROMPT_TYPE)].copy()
manifest = manifest.drop_duplicates(subset=["image_ID", "anchor_idx"], keep="last")

runs = []
for _, row in manifest.iterrows():
    path = os.path.join(RAW_DETECTIONS_DIR, str(row["npz_file"]))
    if not os.path.exists(path):
        print("MISSING npz (skipped):", path)
        continue
    run = load_run_detections(path)
    run["Prompt_ID"] = str(row["Prompt_ID"])
    run["Prompt_Type"] = str(row["Prompt_Type"])
    runs.append(run)

print(f"Loaded {len(runs)} runs "
      f"({manifest['image_ID'].nunique()} images) for {EXPERIMENT_NAME} ({PROMPT_TYPE}).")
print("Total pre-NMS detections:", int(sum(len(r['scores']) for r in runs)))
print("Total GT boxes over all runs:", int(sum(len(r['gt_boxes']) for r in runs)))

In [ ]:
# =============================================================================
# CELL 16 - OFFLINE NMS
# =============================================================================
# SAM3 can return several overlapping boxes for the same plant. NMS keeps the
# highest-scoring box of each overlapping group. The NMS IoU threshold is fixed
# (NMS_IOU_THRESHOLD = 0.40) and applied offline, so the raw pre-NMS detections
# stay untouched on disk.
# (No tile provenance is tracked here: without tiling every detection comes
#  from the same single input image.)
# =============================================================================

def nms(boxes, scores, iou_threshold):
    """
    Input : boxes (N,4), scores (N,), iou_threshold
    Output: keep (list of kept indices, highest score first),
            n_suppressed (int)
    A detection is suppressed when its IoU with an already kept, higher-scoring
    detection is GREATER than the threshold.
    """
    n = len(boxes)
    if n == 0:
        return [], 0
    order = list(np.argsort(-np.asarray(scores, dtype=np.float32), kind="stable"))
    keep, n_suppressed = [], 0
    while order:
        i = int(order[0])
        keep.append(i)
        rest = np.array(order[1:], dtype=int)
        if rest.size == 0:
            break
        ious = compute_iou_matrix(boxes[i:i + 1], boxes[rest])[0]
        n_suppressed += int((ious > iou_threshold).sum())
        order = list(rest[ious <= iou_threshold])
    return keep, n_suppressed


def apply_nms_to_run(run, iou_threshold):
    """
    Apply NMS to one run's pre-NMS detections.

    Output: dict with 'boxes' and 'scores' of the surviving detections SORTED BY
            SCORE (high -> low), plus 'n_pre_nms' and 'n_suppressed'.
    """
    keep, n_suppressed = nms(run["boxes"], run["scores"], iou_threshold)
    keep = np.array(keep, dtype=int)
    return {
        "boxes": run["boxes"][keep].reshape(-1, 4),
        "scores": run["scores"][keep].reshape(-1),
        "n_pre_nms": int(len(run["scores"])),
        "n_suppressed": int(n_suppressed),
    }


print("Offline NMS ready. NMS IoU threshold (fixed):", NMS_IOU_THRESHOLD)

In [ ]:
# =============================================================================
# CELL 17 - EVALUATION CORE: all_gt AND held_out
# =============================================================================
# all_gt   : classical evaluation, every GT box of the image counts.
#
# held_out : the GT instances shown to SAM3 as POSITIVE prompts are REMOVED
#            from the GT set, and predictions that fall on those plants are
#            IGNORED (neither TP nor FP). The positives lie INSIDE the evaluated
#            image and SAM3 usually re-detects them. It answers: "after being
#            shown K examples, how well does SAM3 find the REMAINING Rumex plants?"
#            Negative prompts are not GT boxes and play no role here.
#
# ORDER OF OPERATIONS (the agreed protocol):
#   1. match predictions to the evaluated (non-prompt) GT with the corrected
#      one-to-one matcher at EVAL_IOU_THRESHOLD = 0.50
#   2. every STILL UNMATCHED prediction whose best IoU with a PROMPT GT box is
#      >= PROMPT_IGNORE_IOU (0.50) becomes IGNORED
#   3. whatever is still unmatched is a false positive
#   Prompt GT boxes themselves are never counted as false negatives.
# =============================================================================

STATUS_FP, STATUS_TP, STATUS_IGNORED = 0, 1, 2


def split_gt_for_mode(gt_boxes, prompt_indices, mode):
    """
    Input : all GT boxes of the image, the indices used as visual prompts, mode
    Output: (evaluated_gt_boxes, prompt_gt_boxes)
            all_gt   -> (all boxes, empty)
            held_out -> (non-prompt boxes, prompt boxes)
    """
    gt_boxes = np.asarray(gt_boxes, dtype=np.float32).reshape(-1, 4)
    if mode == "all_gt":
        return gt_boxes, np.zeros((0, 4), dtype=np.float32)
    is_prompt = np.zeros(len(gt_boxes), dtype=bool)
    prompt_indices = np.asarray(prompt_indices, dtype=int)
    if len(prompt_indices):
        is_prompt[prompt_indices] = True
    return gt_boxes[~is_prompt], gt_boxes[is_prompt]


def evaluate_run_predictions(pred_boxes, pred_scores, eval_gt_boxes, prompt_gt_boxes,
                             eval_iou=EVAL_IOU_THRESHOLD, ignore_iou=PROMPT_IGNORE_IOU):
    """
    Evaluate ONE prediction set against ONE GT set.

    Output dict:
      status      (P,) int  - STATUS_TP / STATUS_FP / STATUS_IGNORED per prediction
      TP, FP, FN, n_ignored, n_eval_gt, n_pred
      precision, recall, F1
      IoU1 - mean IoU of the MATCHED prediction/GT pairs only
             ("when it finds a plant, how well is it localised?")
      IoU2 - sum of matched IoUs divided by the number of evaluated GT boxes
             ("localisation quality over ALL plants, missed ones count as 0")
      valid_for_macro - False when there is no GT left to evaluate (held_out runs
             on an image whose only plant was the prompt)
    """
    pred_boxes = np.asarray(pred_boxes, dtype=np.float32).reshape(-1, 4)
    pred_scores = np.asarray(pred_scores, dtype=np.float32).reshape(-1)
    eval_gt_boxes = np.asarray(eval_gt_boxes, dtype=np.float32).reshape(-1, 4)
    prompt_gt_boxes = np.asarray(prompt_gt_boxes, dtype=np.float32).reshape(-1, 4)

    n_pred, n_eval_gt = len(pred_boxes), len(eval_gt_boxes)
    # start with everything as FP -> valid matches become TP -> leftovers on the
    # prompt plant become IGNORED -> the rest stays FP
    status = np.full(n_pred, STATUS_FP, dtype=np.int8)

    # step 1 - corrected one-to-one matching against the evaluated GT
    match = match_one_to_one(pred_boxes, pred_scores, eval_gt_boxes, eval_iou)
    status[match["pred_match_gt"] >= 0] = STATUS_TP

    # step 2 - ignore the unmatched leftovers that sit on a PROMPT plant
    if n_pred and len(prompt_gt_boxes):
        leftover = np.where(status == STATUS_FP)[0]
        if len(leftover):
            best_prompt_iou = compute_iou_matrix(pred_boxes[leftover], prompt_gt_boxes).max(axis=1)
            status[leftover[best_prompt_iou >= ignore_iou]] = STATUS_IGNORED

    tp = int((status == STATUS_TP).sum())
    fp = int((status == STATUS_FP).sum())          # step 3 - the rest are FP
    n_ignored = int((status == STATUS_IGNORED).sum())
    fn = int(n_eval_gt - tp)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / n_eval_gt if n_eval_gt > 0 else 0.0
    f1 = safe_f1(precision, recall)
    matched_ious = match["matched_ious"]
    iou1 = float(np.mean(matched_ious)) if matched_ious else 0.0
    iou2 = float(np.sum(matched_ious) / n_eval_gt) if n_eval_gt > 0 else 0.0

    return {
        "status": status, "pred_match_gt": match["pred_match_gt"],
        "TP": tp, "FP": fp, "FN": fn, "n_ignored": n_ignored,
        "n_eval_gt": n_eval_gt, "n_pred": n_pred,
        "precision": float(precision), "recall": float(recall), "F1": float(f1),
        "IoU1": iou1, "IoU2": iou2,
        "valid_for_macro": bool(n_eval_gt > 0),
    }


print("Evaluation core ready (modes:", EVALUATION_MODES, ")")

In [ ]:
# =============================================================================
# CELL 18 - AP50 AND AP50:95  (supervision.metrics.MeanAveragePrecision)
# =============================================================================
# AP is an area under the precision-recall curve, built by walking through ALL
# detections ordered by confidence. It therefore always uses every post-NMS
# prediction that SAM3 returned (score > CONFIDENCE_THRESHOLD).
# Precision / recall / F1 / IoU1 / IoU2 describe ONE operating point.
# In this experiment the operating point IS the inference threshold (0.30), so
# both use exactly the same prediction set - they still answer different
# questions (ranking quality vs. deployed behaviour).
# =============================================================================

def make_detections(boxes, scores=None):
    """NumPy boxes (+ optional scores) -> supervision Detections (class 0 = Rumex)."""
    boxes = np.asarray(boxes, dtype=np.float32).reshape(-1, 4)
    n = len(boxes)
    if scores is None:
        return sv.Detections(xyxy=boxes, class_id=np.zeros(n, dtype=int))
    return sv.Detections(xyxy=boxes,
                         confidence=np.asarray(scores, dtype=np.float32).reshape(-1),
                         class_id=np.zeros(n, dtype=int))


def compute_ap(pred_list, gt_list):
    """
    Input : two equally long lists of supervision Detections (predictions / GT).
            One entry = one evaluation episode (one run).
    Output: (AP50, AP50_95). NaN when there is no GT at all to evaluate.
    Passing several episodes at once gives the POOLED (dataset-level) AP, in which
    the detections of all episodes are ranked together in one PR curve.
    """
    if len(gt_list) == 0 or sum(len(g) for g in gt_list) == 0:
        return float("nan"), float("nan")
    try:
        result = MeanAveragePrecision().update(pred_list, gt_list).compute()
        ap50, ap5095 = float(result.map50), float(result.map50_95)
        # supervision returns -1.0 when a metric is undefined -> report NaN instead,
        # so it is excluded from means instead of dragging them down.
        return (ap50 if ap50 >= 0 else float("nan"),
                ap5095 if ap5095 >= 0 else float("nan"))
    except Exception as e:
        print("   (AP computation failed:", e, ")")
        return float("nan"), float("nan")


def ap_inputs_for_run(nms_run, eval_gt, prompt_gt):
    """
    Build the (prediction, GT) episode used for AP of ONE run.
    All post-NMS predictions are used; in held_out mode the predictions that were
    IGNORED (they belong to the prompt plant) are removed first, exactly like in
    the operating-point evaluation.
    """
    ev = evaluate_run_predictions(nms_run["boxes"], nms_run["scores"], eval_gt, prompt_gt)
    keep = ev["status"] != STATUS_IGNORED
    return (make_detections(nms_run["boxes"][keep], nms_run["scores"][keep]),
            make_detections(eval_gt))


print("AP helpers ready (AP uses every post-NMS prediction with score > "
      f"{CONFIDENCE_THRESHOLD}).")

In [ ]:
# =============================================================================
# CELL 19 - OPERATING-POINT EVALUATION
# =============================================================================
# Keeps the predictions with score >= CONFIDENCE_THRESHOLD and evaluates them
# (precision, recall, F1, IoU1, IoU2, TP, FP, FN).
# The operating point is fixed from the start - no confidence x NMS sweep and no
# "best configuration" selection is performed in this experiment:
#     confidence threshold = CONFIDENCE_THRESHOLD (0.30)
#     NMS IoU threshold    = NMS_IOU_THRESHOLD    (0.40)
# =============================================================================

def evaluate_at_operating_point(nms_run, eval_gt, prompt_gt,
                                confidence_threshold=CONFIDENCE_THRESHOLD):
    """
    Input : nms_run  - output of apply_nms_to_run (sorted by score, high -> low)
            eval_gt / prompt_gt - from split_gt_for_mode
            confidence_threshold - the operating point
    Output: the dict of evaluate_run_predictions for the thresholded predictions.
    """
    keep = nms_run["scores"] >= confidence_threshold
    return evaluate_run_predictions(nms_run["boxes"][keep], nms_run["scores"][keep],
                                    eval_gt, prompt_gt)


print("Operating configuration (fixed, no sweep):")
print(f"  Confidence threshold = {CONFIDENCE_THRESHOLD:.2f}")
print(f"  NMS IoU threshold    = {NMS_IOU_THRESHOLD:.2f}")

In [ ]:
# =============================================================================
# CELL 20 - RUN-LEVEL METRICS  (one run = one image x one anchor)
# =============================================================================
# Everything is evaluated at the fixed configuration from CELL 3.
#   AP50 / AP50_95 : all post-NMS predictions, confidence-ranked
#   P / R / F1 / IoU1 / IoU2 / TP / FP / FN : predictions >= CONFIDENCE_THRESHOLD
#
# Special case (held_out with no evaluable GT, i.e. the image contains a single
# plant and it was used as the prompt): the metrics are written as NaN and
# valid_for_macro = False so they are excluded from every mean/std, but TP/FN = 0
# and the real FP count are kept, because such a run can still produce false
# positives that must show up in the pooled counts and in the confusion matrix.
# =============================================================================

run_rows = []
for run in runs:
    nms_run = apply_nms_to_run(run, NMS_IOU_THRESHOLD)
    for mode in EVALUATION_MODES:
        eval_gt, prompt_gt = split_gt_for_mode(run["gt_boxes"], run["prompt_indices"], mode)

        # ---- AP: every post-NMS prediction (ignored ones removed) ------------
        p_det, g_det = ap_inputs_for_run(nms_run, eval_gt, prompt_gt)
        ap50, ap5095 = compute_ap([p_det], [g_det])

        # ---- operating point -------------------------------------------------
        ev = evaluate_at_operating_point(nms_run, eval_gt, prompt_gt, CONFIDENCE_THRESHOLD)
        valid = ev["valid_for_macro"]
        nan = float("nan")

        run_rows.append({
            "experiment_name": EXPERIMENT_NAME,
            "image_ID": run["image_ID"],
            "anchor_idx": run["anchor_idx"],
            "Prompt_ID": run["Prompt_ID"],
            "Prompt_Type": run["Prompt_Type"],
            "n_negatives": int(len(run["neg_boxes"])),
            "evaluation_mode": mode,
            "confidence_threshold": CONFIDENCE_THRESHOLD,
            "nms_iou_threshold": NMS_IOU_THRESHOLD,
            "n_gt_total": int(len(run["gt_boxes"])),
            "n_prompt_gt": int(len(run["prompt_indices"])) if mode == "held_out" else 0,
            "n_eval_gt": ev["n_eval_gt"],
            "n_predictions": ev["n_pred"],
            "n_ignored_predictions": ev["n_ignored"],
            "AP50": ap50 if valid else nan,
            "AP50_95": ap5095 if valid else nan,
            "precision": ev["precision"] if valid else nan,
            "recall": ev["recall"] if valid else nan,
            "F1": ev["F1"] if valid else nan,
            "IoU1": ev["IoU1"] if valid else nan,
            "IoU2": ev["IoU2"] if valid else nan,
            "TP": ev["TP"], "FP": ev["FP"], "FN": ev["FN"],
            "valid_for_macro": valid,
        })

run_level_df = pd.DataFrame(run_rows)
RUN_LEVEL_CSV = os.path.join(METRICS_DIR, "run_level_metrics.csv")
run_level_df.to_csv(RUN_LEVEL_CSV, index=False)

print(f"Run-level metrics: {len(run_level_df)} rows -> {RUN_LEVEL_CSV}")
for mode in EVALUATION_MODES:
    sub = run_level_df[run_level_df["evaluation_mode"] == mode]
    print(f"  {mode:9s}: {len(sub)} runs, "
          f"{int(sub['valid_for_macro'].sum())} valid for macro averaging, "
          f"F1_mean={sub['F1'].mean():.4f}")

In [ ]:
# =============================================================================
# CELL 21 - IMAGE-LEVEL METRICS
# =============================================================================
# All anchor runs of the same image are averaged into ONE value per image and per
# evaluation mode. The std here is the spread BETWEEN the different anchors of
# the SAME image, i.e. "how sensitive is the result to which plant was used as
# the visual prompt?".
# NaN rows (held_out runs with no evaluable GT) are ignored by pandas mean/std.
# std is NaN when an image has only one valid run - that is expected.
# =============================================================================

METRIC_COLUMNS = ["AP50", "AP50_95", "precision", "recall", "F1", "IoU1", "IoU2"]

image_rows = []
for (image_id, mode), grp in run_level_df.groupby(["image_ID", "evaluation_mode"]):
    row = {
        "experiment_name": EXPERIMENT_NAME,
        "image_ID": image_id,
        "evaluation_mode": mode,
        "confidence_threshold": CONFIDENCE_THRESHOLD,
        "nms_iou_threshold": NMS_IOU_THRESHOLD,
        "n_runs_total": int(len(grp)),
        "n_runs_valid_for_macro": int(grp["valid_for_macro"].sum()),
        "TP_sum": int(grp["TP"].sum()),
        "FP_sum": int(grp["FP"].sum()),
        "FN_sum": int(grp["FN"].sum()),
    }
    for col in METRIC_COLUMNS:
        row[f"{col}_mean"] = grp[col].mean()      # NaNs skipped automatically
        row[f"{col}_std"] = grp[col].std()        # sample std (ddof=1)
    image_rows.append(row)

image_level_df = pd.DataFrame(image_rows).sort_values(
    ["evaluation_mode", "image_ID"]).reset_index(drop=True)
IMAGE_LEVEL_CSV = os.path.join(METRICS_DIR, "image_level_metrics.csv")
image_level_df.to_csv(IMAGE_LEVEL_CSV, index=False)

print(f"Image-level metrics: {len(image_level_df)} rows -> {IMAGE_LEVEL_CSV}")
print(image_level_df.groupby("evaluation_mode")[
    ["AP50_mean", "precision_mean", "recall_mean", "F1_mean", "IoU1_mean", "IoU2_mean"]
].mean().to_string())

In [ ]:
# =============================================================================
# CELL 22 - EXPERIMENT-LEVEL SUMMARY
# =============================================================================
# Computed from the IMAGE-LEVEL values, not from the raw run rows, so that every
# UAV image contributes exactly the same weight regardless of how many GT boxes
# (and therefore how many anchor runs) it contains.
# The std here is the variation BETWEEN UAV images.
# =============================================================================

summary_rows = []
for mode in EVALUATION_MODES:
    sub = image_level_df[image_level_df["evaluation_mode"] == mode]
    row = {
        "experiment_name": EXPERIMENT_NAME,
        "evaluation_mode": mode,
        "prompt_type": PROMPT_TYPE,
        "n_exemplars": N_EXEMPLARS,
        "n_negatives": J_NEGATIVES,
        "use_tiling": USE_TILING,
        "confidence_threshold": CONFIDENCE_THRESHOLD,
        "nms_iou_threshold": NMS_IOU_THRESHOLD,
        "eval_iou_threshold": EVAL_IOU_THRESHOLD,
        "n_images": int(sub["image_ID"].nunique()),
        "n_runs": int(sub["n_runs_total"].sum()),
        "n_runs_valid_for_macro": int(sub["n_runs_valid_for_macro"].sum()),
    }
    for col in METRIC_COLUMNS:
        row[f"{col}_mean"] = sub[f"{col}_mean"].mean()
        row[f"{col}_std"] = sub[f"{col}_mean"].std()   # spread between images
    summary_rows.append(row)

experiment_summary_df = pd.DataFrame(summary_rows)
EXPERIMENT_SUMMARY_CSV = os.path.join(METRICS_DIR, "experiment_summary.csv")
experiment_summary_df.to_csv(EXPERIMENT_SUMMARY_CSV, index=False)

print(f"Experiment summary -> {EXPERIMENT_SUMMARY_CSV}\n")
print(experiment_summary_df.to_string(index=False))

In [ ]:
# =============================================================================
# CELL 23 - POOLED DATASET AP50 / AP50:95
# =============================================================================
# This is NOT the mean of the image-level AP values. All runs are handed to
# supervision as evaluation EPISODES at once, so every detection of the whole
# dataset is ranked in ONE precision-recall curve.
#
# NOTE for the thesis text: because every GT box of an image becomes an anchor
# once, the same UAV image appears in several episodes (once per anchor).
# The pooled AP is therefore computed over "pooled evaluation episodes", not over
# unique images - it measures the ranking quality of the whole experiment.
# =============================================================================

dataset_rows = []
for mode in EVALUATION_MODES:
    pred_list, gt_list, images_used = [], [], set()
    for run in runs:
        nms_run = apply_nms_to_run(run, NMS_IOU_THRESHOLD)
        eval_gt, prompt_gt = split_gt_for_mode(run["gt_boxes"], run["prompt_indices"], mode)
        p_det, g_det = ap_inputs_for_run(nms_run, eval_gt, prompt_gt)
        pred_list.append(p_det)
        gt_list.append(g_det)
        images_used.add(run["image_ID"])
    ap50, ap5095 = compute_ap(pred_list, gt_list)   # one PR curve over ALL runs
    dataset_rows.append({
        "experiment_name": EXPERIMENT_NAME,
        "evaluation_mode": mode,
        "n_images": len(images_used),
        "n_runs": len(runs),
        "confidence_used_for_AP": CONFIDENCE_THRESHOLD,
        "nms_iou_threshold": NMS_IOU_THRESHOLD,
        "dataset_AP50": ap50,
        "dataset_AP50_95": ap5095,
    })
    del pred_list, gt_list
    gc.collect()

dataset_ap_df = pd.DataFrame(dataset_rows)
DATASET_AP_CSV = os.path.join(METRICS_DIR, "dataset_ap_metrics.csv")
dataset_ap_df.to_csv(DATASET_AP_CSV, index=False)

print(f"Dataset pooled AP -> {DATASET_AP_CSV}\n")
print(dataset_ap_df.to_string(index=False))
print("\nFor comparison, the MEAN of the image-level AP50 values "
      "(a different quantity):")
print(image_level_df.groupby("evaluation_mode")["AP50_mean"].mean().to_string())

In [ ]:
# =============================================================================
# CELL 24 - DATASET-LEVEL CONFUSION MATRICES
# =============================================================================
# One class (Rumex) plus a background row/column:
#     Actual Rumex      -> Predicted Rumex      = TP
#     Actual Rumex      -> Predicted Background = FN  (missed plants)
#     Actual Background -> Predicted Rumex      = FP  (spurious detections)
#     Actual Background -> Predicted Background = not defined for detection
#                                                 (there are no true negatives)
# Counts are pooled over every run at the fixed configuration. In held_out mode
# the prompt plants and the detections that were ignored do not appear anywhere.
# =============================================================================

def plot_confusion_matrix(tp, fp, fn, title, png_path):
    """2x2 detection confusion matrix; the background/background cell stays empty."""
    matrix = np.array([[tp, fn], [fp, np.nan]], dtype=float)
    fig, ax = plt.subplots(figsize=(5.2, 4.6))
    im = ax.imshow(np.nan_to_num(matrix, nan=0.0), cmap="Blues")
    ax.set_xticks([0, 1], ["Predicted\nRumex", "Predicted\nBackground"])
    ax.set_yticks([0, 1], ["Actual\nRumex", "Actual\nBackground"])
    labels = [[f"TP\n{tp}", f"FN\n{fn}"], [f"FP\n{fp}", "n/a\n(no true\nnegatives)"]]
    vmax = np.nanmax(matrix) if np.nanmax(matrix) > 0 else 1.0
    for i in range(2):
        for j in range(2):
            value = matrix[i, j]
            colour = "white" if (not np.isnan(value) and value > 0.5 * vmax) else "black"
            ax.text(j, i, labels[i][j], ha="center", va="center",
                    color=colour, fontsize=11)
    ax.set_title(title, fontsize=11)
    fig.colorbar(im, ax=ax, fraction=0.046)
    fig.tight_layout()
    fig.savefig(png_path, dpi=200)
    plt.show()
    plt.close(fig)


confusion_summary = []
for mode in EVALUATION_MODES:
    sub = run_level_df[run_level_df["evaluation_mode"] == mode]
    tp, fp, fn = int(sub["TP"].sum()), int(sub["FP"].sum()), int(sub["FN"].sum())
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = safe_f1(precision, recall)

    cm_df = pd.DataFrame(
        [[tp, fn], [fp, np.nan]],
        index=["actual_rumex", "actual_background"],
        columns=["predicted_rumex", "predicted_background"],
    )
    csv_path = os.path.join(CONFUSION_MATRIX_DIR, f"confusion_matrix_{mode}.csv")
    cm_df.to_csv(csv_path)

    png_path = os.path.join(CONFUSION_MATRIX_DIR, f"confusion_matrix_{mode}.png")
    plot_confusion_matrix(
        tp, fp, fn,
        f"{EXPERIMENT_NAME} - {mode}\nconf={CONFIDENCE_THRESHOLD:.2f}, "
        f"NMS IoU={NMS_IOU_THRESHOLD:.2f}, eval IoU={EVAL_IOU_THRESHOLD:.2f}",
        png_path)

    confusion_summary.append({
        "experiment_name": EXPERIMENT_NAME, "evaluation_mode": mode,
        "TP": tp, "FP": fp, "FN": fn,
        "precision_micro": precision, "recall_micro": recall, "F1_micro": f1,
        "confidence_threshold": CONFIDENCE_THRESHOLD, "nms_iou_threshold": NMS_IOU_THRESHOLD,
        "eval_iou_threshold": EVAL_IOU_THRESHOLD,
    })
    print(f"{mode:9s}: TP={tp}  FP={fp}  FN={fn}  "
          f"P={precision:.4f}  R={recall:.4f}  F1={f1:.4f}")

confusion_summary_df = pd.DataFrame(confusion_summary)
confusion_summary_df.to_csv(
    os.path.join(CONFUSION_MATRIX_DIR, "confusion_matrix_summary.csv"), index=False)
print("\nConfusion matrices saved to:", CONFUSION_MATRIX_DIR)

In [ ]:
# =============================================================================
# CELL 25 - FINAL OUTPUT SUMMARY
# =============================================================================

print("=" * 78)
print(f"EXPERIMENT {EXPERIMENT_NAME} - FINAL SUMMARY")
print("=" * 78)
print(f"Prompts per run          : {K_POSITIVES} positive + {J_NEGATIVES} negative boxes, no text ({PROMPT_TYPE})")
print(f"Negative generator       : gap >= {NEG_MIN_GAP_PX}px, close <= {NEG_MAX_GAP_FACTOR} x median plant, "
      f"spatially diverse, new per run")
print(f"Tiling                   : {USE_TILING}  (whole image, longest side -> {MAX_DIM}px)")
print(f"Confidence threshold     : {CONFIDENCE_THRESHOLD:.2f} (fixed; SAM3 run once per image x anchor)")
print(f"Mask threshold           : {MASK_THRESHOLD:.2f}")
print(f"NMS IoU threshold        : {NMS_IOU_THRESHOLD:.2f} (fixed)")
print(f"Evaluation IoU           : {EVAL_IOU_THRESHOLD:.2f}")
print(f"Runs / images            : {len(runs)} runs over "
      f"{run_level_df['image_ID'].nunique()} images")
print("-" * 78)
print("EXPERIMENT-LEVEL RESULTS (mean over images, std between images)")
show = ["evaluation_mode", "AP50_mean", "AP50_std", "AP50_95_mean", "precision_mean",
        "recall_mean", "F1_mean", "F1_std", "IoU1_mean", "IoU2_mean"]
print(experiment_summary_df[show].to_string(index=False))
print("-" * 78)
print("POOLED DATASET AP")
print(dataset_ap_df[["evaluation_mode", "dataset_AP50", "dataset_AP50_95"]].to_string(index=False))
print("-" * 78)
print("POOLED CONFUSION COUNTS")
print(confusion_summary_df[["evaluation_mode", "TP", "FP", "FN",
                            "precision_micro", "recall_micro", "F1_micro"]].to_string(index=False))
print("=" * 78)

print("\nFiles written under", RESULTS_ROOT)
for root, dirs, files in os.walk(RESULTS_ROOT):
    depth = root.replace(RESULTS_ROOT, "").count(os.sep)
    print("  " * depth + os.path.basename(root) + "/")
    if os.path.basename(root) == "raw_detections":
        npz_files = [f for f in files if f.endswith(".npz")]
        for f in sorted(files):
            if not f.endswith(".npz"):
                print("  " * (depth + 1) + f)
        print("  " * (depth + 1) + f"[{len(npz_files)} run NPZ files]")
    else:
        for f in sorted(files):
            print("  " * (depth + 1) + f)

In [ ]:
# =============================================================================
# CELL 26 - QUALITATIVE PLOT: BEST IMAGE, GT (left) vs PREDICTIONS (right)
# =============================================================================
# IMAGE SELECTION (image-level AP50, mode = PLOT_EVALUATION_MODE):
#   1. keep only images that have at least PLOT_MIN_GT_BOXES (7) GT boxes
#      and take the one with the highest image-level AP50_mean
#   2. if NO image has 7 GT boxes: keep the images with the HIGHEST number of GT
#      boxes and take the one among them with the highest AP50_mean
#   ties are broken by F1_mean, then by the number of GT boxes.
#
# RUN SHOWN ON THE RIGHT:
#   the image-level AP50 is a mean over all anchors, but a prediction figure shows
#   ONE run -> the anchor of that image with the highest run-level AP50 (ties: F1)
#   is displayed. Its predictions are the post-NMS boxes with
#   score >= CONFIDENCE_THRESHOLD; the positive prompts (green) and the
#   negative prompts (magenta) of that run are drawn dashed.
#
# The image is downscaled for DISPLAY only (PLOT_MAX_DISPLAY_DIM).
# =============================================================================

def _draw_boxes(ax, boxes, color, linewidth=1.5, linestyle="-", labels=None, fontsize=6):
    """Draw [x1,y1,x2,y2] boxes (display coordinates) on a matplotlib axis."""
    for k, (x1, y1, x2, y2) in enumerate(np.asarray(boxes).reshape(-1, 4)):
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False,
                                       edgecolor=color, linewidth=linewidth,
                                       linestyle=linestyle))
        if labels is not None:
            ax.text(x1, y1 - 2, labels[k], color="white", fontsize=fontsize,
                    va="bottom", ha="left",
                    bbox=dict(facecolor=color, edgecolor="none", pad=0.8, alpha=0.85))


def select_image_for_plot(mode=PLOT_EVALUATION_MODE, min_gt=PLOT_MIN_GT_BOXES):
    """
    Output: (image_level_row, selection_rule_text) or (None, reason)
    """
    img_df = image_level_df[image_level_df["evaluation_mode"] == mode].copy()
    n_gt_per_image = (run_level_df[run_level_df["evaluation_mode"] == mode]
                      .groupby("image_ID")["n_gt_total"].first())
    img_df["n_gt"] = img_df["image_ID"].map(n_gt_per_image)
    img_df = img_df[img_df["AP50_mean"].notna() & img_df["n_gt"].notna()]
    if img_df.empty:
        return None, "no image with a valid AP50"
    img_df["n_gt"] = img_df["n_gt"].astype(int)

    candidates = img_df[img_df["n_gt"] >= min_gt]
    if len(candidates):
        rule = (f"highest image-level AP50 among the {len(candidates)} images "
                f"with >= {min_gt} GT boxes")
    else:
        max_gt = int(img_df["n_gt"].max())
        candidates = img_df[img_df["n_gt"] == max_gt]
        rule = (f"no image has >= {min_gt} GT boxes -> highest image-level AP50 among "
                f"the {len(candidates)} image(s) with the most GT boxes ({max_gt})")

    ranked = candidates.sort_values(["AP50_mean", "F1_mean", "n_gt"],
                                    ascending=False, na_position="last")
    print("Top candidates:")
    print(ranked[["image_ID", "n_gt", "AP50_mean", "F1_mean",
                  "precision_mean", "recall_mean"]].head(5).to_string(index=False))
    return ranked.iloc[0], rule


def plot_gt_vs_predictions(mode=PLOT_EVALUATION_MODE):
    img_row, rule = select_image_for_plot(mode)
    if img_row is None:
        print("Nothing to plot:", rule)
        return
    plot_image_id = img_row["image_ID"]

    # ---- best anchor run of that image ---------------------------------------
    img_runs = run_level_df[(run_level_df["evaluation_mode"] == mode) &
                            (run_level_df["image_ID"] == plot_image_id)]
    run_row = img_runs.sort_values(["AP50", "F1"], ascending=False,
                                   na_position="last").iloc[0]
    anchor = int(run_row["anchor_idx"])
    run = next(r for r in runs
               if r["image_ID"] == plot_image_id and r["anchor_idx"] == anchor)

    nms_run = apply_nms_to_run(run, NMS_IOU_THRESHOLD)
    keep = nms_run["scores"] >= CONFIDENCE_THRESHOLD
    pred_boxes, pred_scores = nms_run["boxes"][keep], nms_run["scores"][keep]
    gt_boxes_plot = run["gt_boxes"]
    exemplar_boxes = gt_boxes_plot[np.asarray(run["prompt_indices"], dtype=int)]
    neg_boxes_plot = run["neg_boxes"]

    # ---- open the image, downscale for display only ---------------------------
    image_paths = {r[3]: r[1] for r in valid_images}
    if plot_image_id not in image_paths:
        print("Image file not found for", plot_image_id)
        return
    with Image.open(image_paths[plot_image_id]) as im:
        im = im.convert("RGB")
        w, h = im.size
        s = min(1.0, PLOT_MAX_DISPLAY_DIM / max(w, h))
        disp_w, disp_h = max(1, int(w * s)), max(1, int(h * s))
        display = np.asarray(im.resize((disp_w, disp_h), Image.BILINEAR))
    to_disp = np.array([disp_w / w, disp_h / h, disp_w / w, disp_h / h], dtype=np.float32)

    # ---- figure: GT (left) | predictions (right) -------------------------------
    fig, axes = plt.subplots(1, 2, figsize=(22, 8.5))
    for ax in axes:
        ax.imshow(display)
        ax.axis("off")

    _draw_boxes(axes[0], gt_boxes_plot * to_disp, color="yellow", linewidth=1.5)
    axes[0].set_title(f"Ground truth: {len(gt_boxes_plot)} Rumex boxes", fontsize=12)

    score_labels = [f"{v:.2f}" for v in pred_scores] if PLOT_SHOW_SCORES else None
    _draw_boxes(axes[1], pred_boxes * to_disp, color="red", linewidth=1.5,
                labels=score_labels)
    _draw_boxes(axes[1], exemplar_boxes * to_disp, color="lime", linewidth=2.5,
                linestyle="--")
    _draw_boxes(axes[1], neg_boxes_plot * to_disp, color="magenta", linewidth=2.5,
                linestyle="--")
    axes[1].set_title(
        f"Predictions: {len(pred_boxes)} boxes | anchor = {anchor} | "
        f"POS {run['Prompt_ID']} + {len(neg_boxes_plot)} NEG (no text)\n"
        f"run AP50={run_row['AP50']:.3f}  P={run_row['precision']:.3f}  "
        f"R={run_row['recall']:.3f}  F1={run_row['F1']:.3f}  "
        f"TP={int(run_row['TP'])} FP={int(run_row['FP'])} FN={int(run_row['FN'])}",
        fontsize=12)

    legend_handles = [
        Line2D([0], [0], color="yellow", lw=2, label="ground truth"),
        Line2D([0], [0], color="red", lw=2, label="prediction"),
        Line2D([0], [0], color="lime", lw=2.5, linestyle="--",
               label=f"positive prompts ({len(exemplar_boxes)})"),
        Line2D([0], [0], color="magenta", lw=2.5, linestyle="--",
               label=f"negative prompts ({len(neg_boxes_plot)})"),
    ]
    fig.legend(handles=legend_handles, loc="lower center", ncol=4, fontsize=11,
               frameon=False)
    fig.suptitle(
        f"{EXPERIMENT_NAME} | {plot_image_id} | mode={mode} | "
        f"image-level AP50_mean={img_row['AP50_mean']:.3f} "
        f"(over {int(img_row['n_runs_valid_for_macro'])} anchors)\n"
        f"{K_POSITIVES} pos + {J_NEGATIVES} neg boxes, no text, no tiling (MAX_DIM={MAX_DIM}px) | "
        f"conf={CONFIDENCE_THRESHOLD:.2f}, NMS IoU={NMS_IOU_THRESHOLD:.2f}, "
        f"mask thr={MASK_THRESHOLD:.2f}\nselection: {rule}",
        fontsize=12)
    fig.tight_layout(rect=[0, 0.04, 1, 0.91])

    png_path = os.path.join(
        PLOTS_DIR, f"best_image_{safe_filename(plot_image_id)}_anchor{anchor:03d}_{mode}.png")
    fig.savefig(png_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    print(f"\nSelected image : {plot_image_id} ({len(gt_boxes_plot)} GT boxes)")
    print(f"Selection rule : {rule}")
    print(f"Shown anchor   : {anchor} (best run-level AP50 of this image), "
          f"POS {run['Prompt_ID']} + {len(neg_boxes_plot)} NEG (no text)")
    print(f"Figure saved   : {png_path}")


plot_gt_vs_predictions(PLOT_EVALUATION_MODE)